In [2]:
import numpy as np
from utilities import *

## Setup Hamiltonian

In [3]:
from functools import lru_cache
from scipy.sparse import coo_matrix, csr_matrix

def fixedN_basis(m: int, N: int):
    """List all m-tuples of nonnegative ints summing to N."""
    if m < 1: raise ValueError("m must be >= 1")
    if N < 0: raise ValueError("N must be >= 0")

    basis = []
    occ = [0]*m

    def rec(pos, remaining):
        if pos == m-1:
            occ[pos] = remaining
            basis.append(tuple(occ))
            return
        for n in range(remaining+1):
            occ[pos] = n
            rec(pos+1, remaining-n)

    rec(0, N)
    return basis

class FixedNBosons:
    """
    Fixed total photon number N across m modes.
    Basis: tuples (n1,...,nm) with sum=N.
    Operators: number-conserving hopping a_i^† a_j.
    """
    def __init__(self, m: int, N: int):
        self.m = m
        self.N = N
        self.basis = fixedN_basis(m, N)
        self.dim = len(self.basis)
        self.basis_to_index = {occ: i for i, occ in enumerate(self.basis)}

    def ket(self, occ):
        occ = tuple(occ)
        v = np.zeros(self.dim, dtype=complex)
        v[self.basis_to_index[occ]] = 1.0
        return v

    def pretty_ket(self, psi, tol=1e-12, max_terms=30):
        terms = [(i, amp, self.basis[i]) for i, amp in enumerate(psi) if abs(amp) > tol]
        terms.sort(key=lambda t: abs(t[1]), reverse=True)
        if not terms:
            return "0"
        lines = []
        for i, amp, occ in terms[:max_terms]:
            lines.append(f"{amp:+.6g} |{occ}>")
        if len(terms) > max_terms:
            lines.append(f"... ({len(terms)-max_terms} more terms)")
        return "\n".join(lines)

    def mean_occupations(self, psi):
        # <n_k> from probabilities in occupation basis
        p = np.abs(psi)**2
        nk = np.zeros(self.m, dtype=float)
        for idx, occ in enumerate(self.basis):
            nk += p[idx] * np.array(occ, dtype=float)
        return np.round(nk, 2)

    @lru_cache(maxsize=None)
    def hop(self, i: int, j: int) -> csr_matrix:
        """
        Return sparse matrix for a_i^† a_j (1-indexed modes).
        Acts within fixed-N subspace.
        """
        if not (1 <= i <= self.m and 1 <= j <= self.m):
            raise ValueError("mode indices must be in 1..m")
        if i == j:
            # This is number operator n_i
            rows, cols, data = [], [], []
            for col, occ in enumerate(self.basis):
                data.append(occ[i-1])
                rows.append(col)
                cols.append(col)
            return csr_matrix((data, (rows, cols)), shape=(self.dim, self.dim), dtype=complex)

        rows, cols, data = [], [], []
        for col, occ in enumerate(self.basis):
            nj = occ[j-1]
            if nj == 0:
                continue
            # new occupation: move one photon from j to i
            new_occ = list(occ)
            new_occ[j-1] -= 1
            new_occ[i-1] += 1
            new_occ = tuple(new_occ)
            row = self.basis_to_index[new_occ]

            amp = np.sqrt((occ[i-1] + 1) * occ[j-1])  # bosonic factor
            rows.append(row)
            cols.append(col)
            data.append(amp)

        return csr_matrix((data, (rows, cols)), shape=(self.dim, self.dim), dtype=complex)

In [4]:
hs = FixedNBosons(m=8, N=3)               # 3 photons over 8 modes
psi = hs.ket((3,0,0,0,0,0,0,0))           # all 3 photons in mode 1

print("Initial:")
print(hs.pretty_ket(psi))
print("<n_k> =", hs.mean_occupations(psi))

# Apply a_2^† a_1 : move one photon from mode 1 to mode 2
psi2 = hs.hop(2,1) @ psi
# Normalize if you want to interpret as a state after a non-unitary operator application
psi2 = psi2 / np.linalg.norm(psi2)
print("\nAfter hop (2 <- 1):")
print(hs.pretty_ket(psi2))
print("<n_k> =", hs.mean_occupations(psi2))


Initial:
+1+0j |(3, 0, 0, 0, 0, 0, 0, 0)>
<n_k> = [3. 0. 0. 0. 0. 0. 0. 0.]

After hop (2 <- 1):
+1+0j |(2, 1, 0, 0, 0, 0, 0, 0)>
<n_k> = [2. 1. 0. 0. 0. 0. 0. 0.]


## Second quantization ECM

In [5]:
from scipy.sparse.linalg import expm_multiply

def build_H_edges(hs, edges, eps=None):
    H = 0
    for i, j, Jij in edges:  # 1-indexed modes
        H = H + Jij * hs.hop(i, j) + np.conj(Jij) * hs.hop(j, i)
    if eps is not None:
        for i, ei in enumerate(eps, start=1):
            if ei != 0:
                H = H + ei * hs.hop(i, i)
    return H

def evolve(H, psi0, t):
    return expm_multiply((-1j * t) * H, psi0)

def layer_totals(nk):
    # modes 1-2 = L1, 3-4 = L2, 5-6 = L3
    return np.array([nk[0]+nk[1], nk[2]+nk[3], nk[4]+nk[5]])

def print_layers(label, hs, psi):
    nk = hs.mean_occupations(psi)  # unitary -> normalized
    print(f"{label:>10}  <L1,L2,L3> = {layer_totals(nk)}    <modes> = {nk}")

def K_from_edges(edges, src_modes, dst_modes):
    """
    Build the coupling matrix K (dst x src) from edges (src<->dst only).
    K[dst_index, src_index] = Jij for term a_dst^† a_src (with h.c. added in H builder).
    """
    src_to_col = {m: c for c, m in enumerate(src_modes)}
    dst_to_row = {m: r for r, m in enumerate(dst_modes)}
    K = np.zeros((len(dst_modes), len(src_modes)), dtype=complex)
    for i, j, Jij in edges:
        # interpret edge as undirected; decide orientation by membership
        if (i in src_to_col) and (j in dst_to_row):
            K[dst_to_row[j], src_to_col[i]] += Jij
        elif (j in src_to_col) and (i in dst_to_row):
            K[dst_to_row[i], src_to_col[j]] += Jij
    return K

def choose_time_from_K(K, method="rms"):
    s = np.linalg.svd(K, compute_uv=False)  # singular values
    s = np.sort(s)[::-1]                    # descending
    s_max = s[0]
    s_min = s[-1] if len(s) > 1 else s[0]
    s_rms = np.sqrt(np.mean(s**2))

    if method == "max":
        sigma = s_max
    elif method == "min":
        sigma = s_min
    elif method == "rms":
        sigma = s_rms
    else:
        raise ValueError("method must be one of: 'max', 'min', 'rms'")

    if sigma == 0:
        raise ValueError("All couplings are zero; cannot choose an interaction time.")
    return np.pi / (2 * sigma), s  # return time and the singular values

In [6]:
# ----------------------------
# Setup: 3 layers x 2 modes
# L1: (1,2), L2: (3,4), L3: (5,6)
# ----------------------------
m = 6
N = 2
hs = FixedNBosons(m=m, N=N)

# Initial: one photon per mode in layer 1 => |1,1,0,0,0,0>
psi0 = hs.ket((1,1,0,0,0,0))

# Different couplings between modes (interconnected, non-uniform)
edges12 = [(1,3, 1.0), (1,4, 0.3),
           (2,3, 0.2), (2,4, 0.9)]

edges23 = [(3,5, 0.7), (3,6, 0.2),
           (4,5, 0.4), (4,6, 1.1)]

H12 = build_H_edges(hs, edges12)
H23 = build_H_edges(hs, edges23)

# Choose non-greedy interaction times from coupling spectra
K12 = K_from_edges(edges12, src_modes=[1,2], dst_modes=[3,4])
K23 = K_from_edges(edges23, src_modes=[3,4], dst_modes=[5,6])

t1, s12 = choose_time_from_K(K12, method="rms")  # try "max"/"min"/"rms"
t2, s23 = choose_time_from_K(K23, method="rms")

print("Singular values K12:", s12, "=> t1 =", t1)
print("Singular values K23:", s23, "=> t2 =", t2)

# Evolve step-by-step
psi1 = evolve(H12, psi0, t1)
psi2 = evolve(H23, psi1, t2)

print_layers("initial", hs, psi0)
print_layers("after H12", hs, psi1)
print_layers("after H23", hs, psi2)

# Optional: see how populations change through "steps" by sampling around the chosen times
print("\nStep-1 sweep around t1 (just to observe, not to optimize):")
for frac in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25]:
    psi_tmp = evolve(H12, psi0, frac*t1)
    print(f"t={frac*t1:7.3f}", layer_totals(hs.mean_occupations(psi_tmp)))

print("\nStep-2 sweep around t2 starting from psi1:")
for frac in [0.0, 0.25, 0.5, 0.75, 1.0, 1.25]:
    psi_tmp = evolve(H23, psi1, frac*t2)
    print(f"t={frac*t2:7.3f}", layer_totals(hs.mean_occupations(psi_tmp)))


Singular values K12: [1.20626586 0.6963639 ] => t1 = 1.5949020266171732
Singular values K23: [1.26609364 0.54498339] => t2 = 1.611603026826441
   initial  <L1,L2,L3> = [2. 0. 0.]    <modes> = [1. 1. 0. 0. 0. 0.]
 after H12  <L1,L2,L3> = [0.32 1.69 0.  ]    <modes> = [0.15 0.17 0.85 0.84 0.   0.  ]
 after H23  <L1,L2,L3> = [0.32 0.51 1.18]    <modes> = [0.15 0.17 0.3  0.21 0.56 0.62]

Step-1 sweep around t1 (just to observe, not to optimize):
t=  0.000 [2. 0. 0.]
t=  0.399 [1.71 0.28 0.  ]
t=  0.797 [1.05 0.95 0.  ]
t=  1.196 [0.47 1.53 0.  ]
t=  1.595 [0.32 1.69 0.  ]
t=  1.994 [0.58 1.42 0.  ]

Step-2 sweep around t2 starting from psi1:
t=  0.000 [0.32 1.69 0.  ]
t=  0.403 [0.32 1.44 0.25]
t=  0.806 [0.32 0.9  0.78]
t=  1.209 [0.32 0.51 1.17]
t=  1.612 [0.32 0.51 1.18]
t=  2.015 [0.32 0.77 0.92]


## Add ancilla atom detectors

In [7]:
from scipy.sparse import kron, identity, csr_matrix
from scipy.sparse.linalg import expm_multiply

In [8]:
def separate_subsystems(psi_full, dim_boson, dim_ancilla):
    """
    Given a joint state vector psi_full of dimension dim_boson*dim_ancilla,
    return the reduced density matrices for the bosonic and ancilla subsystems.
    If the state is a product state, also return the corresponding pure state
    vectors up to a global phase.

    Parameters
    ----------
    psi_full : 1D numpy array, shape (dim_boson*dim_ancilla,)
        The full state vector of the composite system.
    dim_boson : int
        Dimension of the bosonic subsystem.
    dim_ancilla : int
        Dimension of the ancilla subsystem (2 in your case).

    Returns
    -------
    rho_b : 2D numpy array, shape (dim_boson, dim_boson)
        Reduced density matrix of the bosonic subsystem.
    rho_a : 2D numpy array, shape (dim_ancilla, dim_ancilla)
        Reduced density matrix of the ancilla subsystem.
    psi_b : 1D numpy array or None
        Pure state vector of the bosonic subsystem if separable, else None.
    psi_a : 1D numpy array or None
        Pure state vector of the ancilla subsystem if separable, else None.
    """
    # reshape into matrix form (rows index boson states, columns index ancilla)
    psi_mat = psi_full.reshape(dim_boson, dim_ancilla)

    # reduced density matrices by partial trace
    rho_b = np.einsum('ik,jk->ij', psi_mat, psi_mat.conj())
    rho_a = np.einsum('ki,kj->ij', psi_mat, psi_mat.conj())

    # attempt to extract pure states if rank-1
    psi_b = None
    psi_a = None
    # eigen-decomposition of rho_b
    vals_b, vecs_b = np.linalg.eigh(rho_b)
    # find the eigenvector with the largest eigenvalue
    if np.isclose(vals_b[-1], 1.0, atol=1e-10):
        psi_b = vecs_b[:, -1]
        # similar for ancilla
        vals_a, vecs_a = np.linalg.eigh(rho_a)
        psi_a = vecs_a[:, -1]
        # fix global phase: make the inner product with psi_mat positive real
        phase = (psi_mat @ psi_a).conj().dot(psi_b)
        psi_b *= np.exp(-1j * np.angle(phase))
    return rho_b, rho_a, psi_b, psi_a

In [9]:
# ancilla Pauli matrices (dense, 2×2)
sigma_z = csr_matrix(np.array([[1, 0], [0, -1]], dtype=complex))
sigma_x = csr_matrix(np.array([[0, 1], [1, 0]], dtype=complex))
sigma_y = csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=complex))
identity_qubit = csr_matrix(np.eye(2, dtype=complex))

In [10]:
hs = FixedNBosons(m=2, N=2)
H_boson = hs.hop(2, 1) + hs.hop(1, 2)  # simple hopping between two modes
psi_boson = hs.ket((2, 0))  # one photon in mode 1

In [11]:
# identities on bosonic space
identity_boson = identity(hs.dim, format='csr', dtype=complex)

In [12]:
# choose which mode to monitor (e.g., k=1) and its number operator
k = 1
n_k = hs.hop(k, k)  # this is a sparse operator acting on the bosonic Hilbert space

In [13]:
# dispersive parameters
chi = 0.05  # dispersive shift per photon (arbitrary units)
omega_q = 0.0  # qubit frequency; set to zero if you only care about the phase

In [14]:
# Build the full Hamiltonian on the tensor product space
H_boson_full = kron(H_boson, identity_qubit)
H_qubit_full = 0.5 * omega_q * kron(identity_boson, sigma_z)
H_disp = chi * kron(n_k, sigma_z)

In [15]:
psi_anc_plus  = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)

In [16]:
t_hop = np.pi / (2 * 1.0)  # time to swap between modes 1 and 2
psi_full = np.kron(psi_boson, psi_anc_plus)  # initial state of boson + ancilla
psi_full_after_hop = expm_multiply((-1j * t_hop) * H_boson_full, psi_full)

In [17]:
rho_b, rho_a, psi_b, psi_a = separate_subsystems(psi_full_after_hop, hs.dim, 2)

In [18]:
hs.mean_occupations(psi_b)

array([0., 2.])

In [19]:
def expectation_full_state(psi, O):
    """
    Expectation value <psi| O |psi>
    """
    return np.vdot(psi, O @ psi)

In [20]:
I_b = identity(hs.dim, format='csr', dtype=complex)

X_full = kron(I_b, sigma_x)
Y_full = kron(I_b, sigma_y)
Z_full = kron(I_b, sigma_z)

In [21]:
# apply int
N = 1  # max photon number 
t_int = 0.1 / (chi * N) 
psi_after_int = expm_multiply((-1j * t_int) * H_disp, psi_full_after_hop)

In [22]:
expectation_full_state(psi_after_int, Y_full)

np.complex128(1.8746159464641201e-31+0j)

In [23]:
rho_b, rho_a, psi_b, psi_a = separate_subsystems(psi_after_int, hs.dim, 2)
hs.mean_occupations(psi_b)

array([0., 2.])

### compressed code for testing

In [24]:
# one layer bosonic ECM
m = 2
N = 3
hs = FixedNBosons(m=m, N=N)
H_boson = hs.hop(2, 1) + hs.hop(1, 2) 
identity_boson = identity(hs.dim, format='csr', dtype=complex) 
psi_boson = hs.ket((2, 1))  

# ancilla
k = 2 # monitor mode 2
n_k = hs.hop(k, k)
chi = 0.05
omega_q = 0.0
H_ancilla = 0.5 * omega_q * sigma_z 
identity_qubit = identity(2, format='csr', dtype=complex)
psi_anc_plus  = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)

# total hamiltonians
H_int = chi * kron(n_k, sigma_z)
H_ecm = kron(H_boson, identity_qubit) + kron(identity_boson, H_ancilla)
psi_full = np.kron(psi_boson, psi_anc_plus)  # initial state of boson + ancilla

In [25]:
t_hop = np.pi / (2 * 1.0)  # time to swap between modes 1 and 2
psi_full = np.kron(psi_boson, psi_anc_plus)  # initial state of boson + ancilla
psi_full_after_hop = expm_multiply((-1j * t_hop) * H_ecm, psi_full)

# state before
print("Initial state, occupations numbers:", hs.mean_occupations(psi_boson))
rho_b, rho_a, psi_b, psi_a = separate_subsystems(psi_full_after_hop, hs.dim, 2)
print("After hop, occupations numbers:", hs.mean_occupations(psi_b))

Initial state, occupations numbers: [2. 1.]
After hop, occupations numbers: [1. 2.]


In [26]:
t_int = 0.1 / (chi * 2)
psi_after_int = expm_multiply((-1j * t_int) * H_int, psi_full_after_hop)

In [27]:
Y_full = kron(identity_boson, sigma_y)

In [28]:
print("ancilla expectation <Y> after int:", np.real(np.round(expectation_full_state(psi_after_int, Y_full), 2) ))

ancilla expectation <Y> after int: 0.2


In [29]:
rho_b, rho_a, psi_b, psi_a = separate_subsystems(psi_after_int, hs.dim, 2)
print("After int, occupations numbers:", hs.mean_occupations(psi_b))

After int, occupations numbers: [1. 2.]


## Multilayer with a few ancillas

In [30]:
# ----------------------------
# Setup: 3 layers x 2 modes
# L1: (1,2), L2: (3,4), L3: (5,6)
# ----------------------------
m = 6
N = 2
hs = FixedNBosons(m=m, N=N)

# Initial: one photon per mode in layer 1 => |1,1,0,0,0,0>
psi0 = hs.ket((1,1,0,0,0,0))

# Different couplings between modes (interconnected, non-uniform)
edges12 = [(1,3, 1.0), (1,4, 0.3),
           (2,3, 0.2), (2,4, 0.9)]

edges23 = [(3,5, 0.7), (3,6, 0.2),
           (4,5, 0.4), (4,6, 1.1)]

H12 = build_H_edges(hs, edges12)
H23 = build_H_edges(hs, edges23)

# Choose non-greedy interaction times from coupling spectra
K12 = K_from_edges(edges12, src_modes=[1,2], dst_modes=[3,4])
K23 = K_from_edges(edges23, src_modes=[3,4], dst_modes=[5,6])

t1, s12 = choose_time_from_K(K12, method="rms")  # try "max"/"min"/"rms"
t2, s23 = choose_time_from_K(K23, method="rms")

In [108]:
# dispersive parameters
chi = 0.05  # dispersive shift per photon (arbitrary units)
omega_q = 0.0  # qubit frequency; set to zero if you only care about the phase

In [109]:
# build ancillas
layers = [2]
modes_per_layer = {2: (3,4)}
H_ancilla = 0.5 * omega_q * sigma_z 
H_hop = {1:H12, 2:H23}
H_L_disp = {}
H_L_hop = {}
for l in layers:
    H_disp = csr_matrix(np.zeros((hs.dim*2**len(modes_per_layer[l]), (hs.dim*2**len(modes_per_layer[l]))), dtype=complex))  # placeholder for full Hamiltonian
    for i, k in enumerate(modes_per_layer[l]):
        # place sigma_z on the correct ancilla qubit
        if i == 0:
                sigma_z_full = sigma_z
        else:
            for j in range(i):
                if j == 0:
                    sigma_z_full = identity_qubit
                else:
                    sigma_z_full = kron(identity_qubit, sigma_z_full)
            sigma_z_full = kron(sigma_z, sigma_z_full)  # pad with identities for remaining ancillas
        for j in range(i+1, len(modes_per_layer[l])):
            sigma_z_full = kron(identity_qubit, sigma_z_full)
        n_k = hs.hop(k, k)
        H_disp += chi * kron(n_k, sigma_z_full) 
    H_L_disp[l] = H_disp
    H_ancillas = np.eye(2**len(modes_per_layer[l]), dtype=complex)
    H_L_hop[1] = kron(H_hop[1], H_ancillas)

In [110]:
psi_anc_plus  = np.array([1/np.sqrt(2), 1/np.sqrt(2)], dtype=complex)

In [111]:
psi_full = np.kron(psi0, np.kron(psi_anc_plus, psi_anc_plus))  # initial state of boson + ancillas
psi_full_after_hop1 = expm_multiply((-1j * t1) * H_L_hop[1], psi_full)

# state before
print("Initial state, occupations numbers:", hs.mean_occupations(psi0))
rho_b, rho_a, psi_b, psi_a = separate_subsystems(psi_full_after_hop1, hs.dim, 2**len(modes_per_layer[2]))
print("After hop1, occupations numbers:", hs.mean_occupations(psi_b))

Initial state, occupations numbers: [1. 1. 0. 0. 0. 0.]
After hop1, occupations numbers: [0.15 0.17 0.85 0.84 0.   0.  ]


In [113]:
t_int = 0.1 / (chi * 2)
psi_after_int = expm_multiply((-1j * t_int) * H_L_disp[2], psi_full_after_hop1)

In [114]:
identity_boson = identity(hs.dim, format='csr', dtype=complex) 
Y_ancilla_1 = kron(identity_boson, kron(sigma_y, identity_qubit))  # Y on ancilla of layer 1
Y_ancilla_2 = kron(identity_boson, kron(identity_qubit, sigma_y))  # Y on ancilla of layer 2
print("ancilla 1 expectation <Y> after int:", np.real(np.round(expectation_full_state(psi_after_int, Y_ancilla_1), 2) ))
print("ancilla 2 expectation <Y> after int:", np.real(np.round(expectation_full_state(psi_after_int, Y_ancilla_2), 2) ))

ancilla 1 expectation <Y> after int: 0.08
ancilla 2 expectation <Y> after int: 0.08
